# Analysis of data for the Essential FFPE Panel

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.metrics import cohen_kappa_score
from sklearn.neighbors import NearestNeighbors
from scipy.stats import gaussian_kde
from sklearn.mixture import GaussianMixture

from tqdm import trange

### BatchDetect module imports

Import project-specific helpers from the `batchdetect` package:

- `load_thal_cross_lot_covs` and related loaders for Thalassemia data.
- `HeavyMixture` and `parametric_bootstrap_lrt` for mixture modeling and
  parametric bootstrap-based likelihood ratio tests.
- Correlation-based clustering utilities:
  `cluster_hierarchical_corr`, `cluster_spectral_corr`,
  `cluster_pca_kmeans_corr`.
- Correlation preprocessing helpers: `normalize_mat` and `get_correlations`.

These functions implement the main batch-detection and clustering methods used
throughout the analysis.


In [2]:
from batchdetect.loader import load_essential_ffpe
from batchdetect.mixture import HeavyMixture,parametric_bootstrap_lrt

df_thal_likelihoods = pd.read_csv('../likelihoods_essential.csv')
snames = df_thal_likelihoods['Sample Name'].values
likelihoods = df_thal_likelihoods['Likelihood'].values
y_hat = df_thal_likelihoods['Label'].values

counts_thal,y_thal,sample_id,features,_ = load_essential_ffpe()
counts_thal = counts_thal[:-1]
y_thal = y_thal[:-1]
sample_id = sample_id[:-1]

index_homozygous_del = []
counts_thal_new = np.delete(counts_thal, index_homozygous_del, axis=0)
likelihoods_new = np.delete(likelihoods, index_homozygous_del)
y_new = np.delete(y_hat, index_homozygous_del)
snames_new = np.delete(snames,index_homozygous_del)

In [3]:
def get_results(dist):
    def null_factory():
        return HeavyMixture(
                n_components=1,
                component_distribution=dist,
                n_init=3,
                max_iter=1000,
            )   

    def alt_factory():
        return HeavyMixture(
                n_components=2,
                component_distribution=dist,
                n_init=3,
                max_iter=1000,
            )   
    res = parametric_bootstrap_lrt(
            likelihoods_new,  
            null_model_factory=null_factory,
            alt_model_factory=alt_factory,
            n_bootstrap=10000,
            random_state=2021,
        )
    return res

In [4]:
res_gaussian = get_results('gaussian')
res_laplace = get_results('laplace')
res_student_t = get_results('student_t')
res_hypsecant = get_results('hypsecant')
res_gennorm = get_results('gennorm')


In [5]:
print("Gaussian p-value: %0.3f"%res_gaussian['p_value'])
print("Laplace p-value: %0.3f"%res_laplace['p_value'])
print("Student-T p-value: %0.3f"%res_student_t['p_value'])
print("Hyp p-value: %0.3f"%res_hypsecant['p_value'])
print("Gennorm p-value: %0.3f"%res_gennorm['p_value'])

Gaussian p-value: 0.013
Laplace p-value: 0.011
Student-T p-value: 0.031
Hyp p-value: 0.007
Gennorm p-value: 0.016


In [6]:
def get_results_subset(dist):
    def null_factory():
        return HeavyMixture(
                n_components=1,
                component_distribution=dist,
                n_init=3,
                max_iter=1000,
            )   

    def alt_factory():
        return HeavyMixture(
                n_components=2,
                component_distribution=dist,
                n_init=3,
                max_iter=1000,
            )   
    res1 = parametric_bootstrap_lrt(
            likelihoods_new[y_new==1],  
            null_model_factory=null_factory,
            alt_model_factory=alt_factory,
            n_bootstrap=10000,
            random_state=2021,
        )
    res2 = parametric_bootstrap_lrt(
            likelihoods_new[y_new==0],  
            null_model_factory=null_factory,
            alt_model_factory=alt_factory,
            n_bootstrap=10000,
            random_state=2021,
        )
    return res1,res2

In [8]:
res_gaussian_subgroup1,res_gaussian_subgroup2 = get_results_subset('gaussian')
res_laplace_subgroup1,res_laplace_subgroup2 = get_results_subset('laplace')
res_student_t_subgroup1,res_student_t_subgroup2 = get_results_subset('student_t')
res_hypsecant_subgroup1,res_hypsecant_subgroup2 = get_results_subset('hypsecant')
res_gennorm_subgroup1,res_gennorm_subgroup2 = get_results_subset('gennorm')


In [9]:
print("Gaussian p-value: %0.3f"%res_gaussian_subgroup1['p_value'])
print("Laplace p-value: %0.3f"%res_laplace_subgroup1['p_value'])
print("Student-T p-value: %0.3f"%res_student_t_subgroup1['p_value'])
print("Hyp p-value: %0.3f"%res_hypsecant_subgroup1['p_value'])
print("Gennorm p-value: %0.3f"%res_gennorm_subgroup1['p_value'])

Gaussian p-value: 0.352
Laplace p-value: 0.233
Student-T p-value: 0.303
Hyp p-value: 0.682
Gennorm p-value: 0.261


In [10]:
print("Gaussian p-value: %0.3f"%res_gaussian_subgroup2['p_value'])
print("Laplace p-value: %0.3f"%res_laplace_subgroup2['p_value'])
print("Student-T p-value: %0.3f"%res_student_t_subgroup2['p_value'])
print("Hyp p-value: %0.3f"%res_hypsecant_subgroup2['p_value'])
print("Gennorm p-value: %0.3f"%res_gennorm_subgroup2['p_value'])

Gaussian p-value: 0.499
Laplace p-value: 0.141
Student-T p-value: 0.294
Hyp p-value: 0.108
Gennorm p-value: 0.264


In [11]:
results = {}
import pickle
results['res_gaussian'] = res_gaussian
results['res_laplace'] = res_laplace
results['res_student_t'] = res_student_t
results['res_hypsecant'] = res_hypsecant
results['res_gennorm'] = res_gennorm

results['res_gaussian_subgroup1'] = res_gaussian_subgroup1
results['res_laplace_subgroup1'] = res_laplace_subgroup1
results['res_student_t_subgroup1'] = res_student_t_subgroup1
results['res_hypsecant_subgroup1'] = res_hypsecant_subgroup1
results['res_gennorm_subgroup1'] = res_gennorm_subgroup1

results['res_gaussian_subgroup2'] = res_gaussian_subgroup2
results['res_laplace_subgroup2'] = res_laplace_subgroup2
results['res_student_t_subgroup2'] = res_student_t_subgroup2
results['res_hypsecant_subgroup2'] = res_hypsecant_subgroup2
results['res_gennorm_subgroup2'] = res_gennorm_subgroup2

with open('LBX_Res_P.p','wb') as f:
    pickle.dump(results,f)